<a href="https://colab.research.google.com/github/Mshaheereijaz/FlyRank-AI-Task/blob/main/notebook02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


**1. A rule you write by hand: stale x visible**

In [29]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")


Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


**2. Let a model learn the rule — then read it**

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")


Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


**3. Why you can't feed the outcome back in**

In [27]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



# TASK



---


***Task: Change max_depth to 3 or 4 — does Precision@50 improve? Can you still read the tree?**

---



In [26]:
from sklearn.tree import DecisionTreeClassifier, export_text

feat = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X2 = df[feat].replace([np.inf, -np.inf], np.nan).fillna(0)

tree2 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree2.fit(X2, y)

print(export_text(tree2, feature_names=feat))

tree2_score = tree2.predict_proba(X2)[:, 1]
print(f"max_depth=3  Precision@50: {precision_at_k(tree2_score, y, 50):.3f}")

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0

max_depth=3  Precision@50: 0.720




---


***Depth comparision (max_depth=2 vs max_depth=3)***


---
Increasing the "max_depth" from 2 to 3 has increased the "Precision@50" from 0.600 to 0.720. However, the tree has become more complex since the depth has increased. It has added more splits which is making it harder to read at first look.



---


***Task : Swap in different features (drop impressions_90d, add engagement_rate). What does the tree choose to split on first?***

---



In [17]:
from sklearn.tree import DecisionTreeClassifier, export_text

ft = ["content_age_days", "days_since_last_update", "engagement_rate",
            "avg_position", "ctr", "word_count"]
X2 = df[ft].replace([np.inf, -np.inf], np.nan).fillna(0)

tree2 = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree2.fit(X2, y)

print(export_text(tree2, feature_names=ft))

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0



After Dropping "impressions_90d" and going for "engagement_rate" the tree choose to split on "avg_postion" rather than "engaement_rate". which means that "avg_position" and "content_age_days" were more useful.



---
***Task : Re-run your comparison with a train/test split and see if the gap holds***

---



In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

tree_split = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_split.fit(X_train, y_train)

test_indices = X_test.index
hand_rule_test = df.loc[test_indices, "hand_rule_score"]
tree_test_score = tree_split.predict_proba(X_test)[:, 1]

for k in (20, 50):
    k_adj = min(k, len(y_test))
    hr = precision_at_k(hand_rule_test, y_test, k_adj)
    tr = precision_at_k(tree_test_score, y_test, k_adj)
    print(f"Test Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Test Precision@20:  hand rule 0.700   vs   tree 0.650
Test Precision@50:  hand rule 0.580   vs   tree 0.680


**Train/test split: does the gap hold?**

Before (using all data to both train and test): hand rule beat the tree at both k=20 (0.900 vs 0.550) and k=50 (0.680 vs 0.600).

After (training on one part, testing on unseen data): hand rule dropped at k=20 (0.900 → 0.700) and k=50 (0.680 → 0.580), while the tree actually improved at k=50 (0.600 → 0.680) and beat the hand rule there.

So the gap does not fully hold. The hand rule looked stronger before mainly because it was being tested on data it already "saw." Once tested on new, unseen pages, the tree does just as well or better, especially at k=50.